# Qwen3.5-4B: local loading experiment

Load the exact competition model `Qwen/Qwen3.5-4B` with **4-bit NF4 weights** on the local RTX 4060 (8 GB). This is a quantized development configuration, not a reproduction of full-precision robustness labels.

From the repository root:
```bash
bash scripts/install_fast_kernels.sh
uv run python -m ipykernel install --user --name aimo-interp-uv --display-name "AIMO Interp (uv)"
```
Select **AIMO Interp (uv)** as the notebook kernel. The first load downloads weights into the ignored `data/huggingface` directory; later loads reuse them. Run cells in order. Restart the kernel to release the model's GPU memory.

[Official model card](https://huggingface.co/Qwen/Qwen3.5-4B) · [Quantization documentation](https://huggingface.co/docs/transformers/quantization/bitsandbytes)


If loading appears stuck, the weight progress bar and a Python stack dump after 60 seconds help locate the wait. Loading uses the already downloaded cache without network requests. Do not rerun the loading cell while a model is resident; restart the kernel first. Generation uses `stream=False` for timing; the complete response is printed after generation finishes.


The setup script installs Flash Linear Attention and builds `causal-conv1d` using uv-managed CUDA tools. Restart the notebook kernel after installation so Transformers detects them. Loading prints `DeltaNet fast path: True` when both optimized kernels are available. The first inference may take longer while Triton compiles kernels.


In [1]:
from pathlib import Path
import sys
import importlib.metadata
from functools import partial

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from aimo_interp.model import CACHE_DIR, MODEL_ID, REVISION, load_model, solve as generate_response
import torch

assert Path(sys.prefix).resolve() == (REPO_ROOT / ".venv").resolve(), "Select the repository's uv kernel."
assert torch.cuda.is_available(), "A CUDA GPU is required for this notebook."
print("Python:", sys.executable)
for package in ["torch", "transformers", "accelerate", "bitsandbytes"]:
    print(f"{package}: {importlib.metadata.version(package)}")
print("GPU:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"VRAM: {free / 2**30:.2f} GiB free / {total / 2**30:.2f} GiB total")
print("Model cache:", CACHE_DIR)


/home/suhas-shankar/Projects/AI_Research/AIMO_Interp/aimo-interp-investigate-black-box/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: /home/suhas-shankar/Projects/AI_Research/AIMO_Interp/aimo-interp-investigate-black-box/.venv/bin/python
torch: 2.12.1
transformers: 5.13.0
accelerate: 1.13.0
bitsandbytes: 0.50.2
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 7.51 GiB free / 7.62 GiB total
Model cache: /home/suhas-shankar/Projects/AI_Research/AIMO_Interp/aimo-interp-investigate-black-box/data/huggingface


Model loading, cast-warning filtering, and prompt scaffolding are defined in `../src/aimo_interp/model.py`.

In [2]:
QUANTIZATION_BITS = 4  # NF4 with double quantization; restart the kernel before switching.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


In [3]:
if "model" in globals():
    raise RuntimeError("Model already loaded. Reuse it, or restart the kernel before loading again.")

model, tokenizer = load_model(
    quantization_bits=QUANTIZATION_BITS,
    compute_dtype=COMPUTE_DTYPE,
)


{'model': 'Qwen/Qwen3.5-4B', 'revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a', 'quantization_bits': 4, 'compute_dtype': 'torch.bfloat16'}
DeltaNet fast path: True
Loading tokenizer...
Loading and quantizing weights on GPU...


Loading weights: 100%|██████████| 723/723 [00:02<00:00, 267.04it/s]


Loaded in 3.5 seconds
Model footprint: 3.01 GiB
CUDA allocated: 3.08 GiB
Parameter devices: ['cuda:0']


## Short generation check

Use a single instruction asking for a boxed final answer, appended to the unchanged problem. `solve()` retains thinking by default for mathematical experiments. The simple 2 + 2 formatting check explicitly disables thinking to isolate output formatting. It is not a competition-model robustness evaluation.

Streaming is disabled for timing. Rerun the math generation cell after the first inference to measure warmed-up performance; timing includes prompt processing. `hit_token_limit=True` indicates possible truncation; inspect whether a final answer was actually produced. Greedy decoding remains a local development setting rather than the organizers' sampled evaluation protocol.


In [4]:
# Reload the helper without reloading the model already on the GPU.
import importlib
from aimo_interp import model as model_module

importlib.reload(model_module)
solve = partial(model_module.solve, model=model, tokenizer=tokenizer)

# Formatting smoke test only; real math runs retain thinking by default.
result = solve("What is 2 + 2?", max_new_tokens=128, enable_thinking=False, stream=False)
print({key: value for key, value in result.items() if key != "response"})
print(result["response"])


Generating...
{'generated_tokens': 32, 'hit_token_limit': False, 'elapsed_seconds': 1.89, 'tokens_per_second': 16.95, 'peak_allocated_gib': 3.13}
The problem asks for the sum of 2 and 2.

$$2 + 2 = 4$$

\boxed{4}



## Non-robust Qwen cases from train-main-v2

These cells can run independently of the GPU/model-loading cells above.

We pin `aimo-interp/train-main-v2` to revision `23791d9b120132087877b0459f953d5959fc8455`. Filter the exact competition model and **explicit False labels**, preserving null labels in the full table. `max_drop` is the largest accuracy decrease across perturbation families, expressed as a fraction: 0.27 means 27 percentage points.

The dataset contains original problems and aggregate robustness metadata, not the perturbations, reference answers, or model solutions. These organizer labels refer to the evaluated model configuration; they do not establish robustness of our locally quantized checkpoint. A correct answer to an original problem alone does not reproduce its robustness label.

[Dataset card at this revision](https://huggingface.co/datasets/aimo-interp/train-main-v2/blob/23791d9b120132087877b0459f953d5959fc8455/README.md)


Download once from the repository root:
```bash
uv run scripts/download_train_main_v2.py
```
The cells below read `data/train-main-v2/data/train-00000-of-00001.parquet` without downloading data.


In [5]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

DATASET_ID = "aimo-interp/train-main-v2"
DATASET_REVISION = "23791d9b120132087877b0459f953d5959fc8455"
DATASET_MODEL_ID = "Qwen/Qwen3.5-4B"
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
dataset_dir = repo_root / "data" / "train-main-v2"
dataset_path = dataset_dir / "data" / "train-00000-of-00001.parquet"
if not dataset_path.is_file():
    raise FileNotFoundError("Run `uv run scripts/download_train_main_v2.py` from the repository root first.")
provenance = json.loads((dataset_dir / "metadata.json").read_text())
if provenance["dataset"] != DATASET_ID or provenance["revision"] != DATASET_REVISION:
    raise ValueError("Dataset revision mismatch. Download the pinned revision with the script.")
train_main_v2 = pd.read_parquet(dataset_path)
train_main_v2.index.name = "dataset_row"  # Positional row, not an official case ID.
qwen_cases = train_main_v2.loc[train_main_v2["model_id"].eq(DATASET_MODEL_ID)].copy()
# Nullable BooleanDtype ensures null labels stay null rather than becoming False.
qwen_cases["is_robust"] = qwen_cases["is_robust"].astype("boolean")
qwen_non_robust = qwen_cases.loc[qwen_cases["is_robust"].eq(False).fillna(False)].copy()
print(f"Dataset: {DATASET_ID}@{DATASET_REVISION}")
print(f"Total dataset rows: {len(train_main_v2)}; Qwen rows: {len(qwen_cases)}")
print(f"Qwen: {qwen_cases['is_robust'].sum()} robust, {len(qwen_non_robust)} non-robust, "
      f"{qwen_cases['is_robust'].isna().sum()} unlabeled")
display(qwen_non_robust[["model_id", "reasoning_effort", "is_robust", "max_drop"]])


Dataset: aimo-interp/train-main-v2@23791d9b120132087877b0459f953d5959fc8455
Total dataset rows: 82; Qwen rows: 14
Qwen: 8 robust, 4 non-robust, 2 unlabeled


,model_id,reasoning_effort,is_robust,max_drop
dataset_row,,,,
10,Qwen/Qwen3.5-4B,default,False,0.27
11,Qwen/Qwen3.5-4B,default,False,0.27
12,Qwen/Qwen3.5-4B,default,False,0.40
13,Qwen/Qwen3.5-4B,default,False,0.68


In [6]:
for row_index, case in qwen_non_robust.iterrows():
    display(Markdown(
        f"### Dataset row {row_index} — max drop {case['max_drop']:.0%}\n\n"
        f"Model: `{case['model_id']}` · Effort: `{case['reasoning_effort']}` · "
        f"Label: **non-robust**\n\n" + case["problem"]
    ))


### Dataset row 10 — max drop 27%

Model: `Qwen/Qwen3.5-4B` · Effort: `default` · Label: **non-robust**

Let $S$ denote the value of the infinite sum
\[
\frac{1}{9} + \frac{1}{99} + \frac{1}{999} + \frac{1}{9999}  + \cdots
\]
Find the remainder when the greatest integer less than or equal to $10^{100} S$ is divided by $1000.$

### Dataset row 11 — max drop 27%

Model: `Qwen/Qwen3.5-4B` · Effort: `default` · Label: **non-robust**

Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by
\begin{equation*}
    f(n) = \sum_{i = 1}^n \sum_{j = 1}^n j^{3272} \left\lfloor\frac1j + \frac{n-i}{n}\right\rfloor.
\end{equation*}
Let $M=2 \cdot 3 \cdot 5 \cdot 7 \cdot 11 \cdot 13$ and let $N = f{\left(M^{7}\right)} - f{\left(M^{7}-1\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $20^3$?

### Dataset row 12 — max drop 40%

Model: `Qwen/Qwen3.5-4B` · Effort: `default` · Label: **non-robust**

Let $N$ be the number of positive integer divisors of $17017^{17}$ that leave a remainder of $5$ when divided by $12$. Find $N$.

### Dataset row 13 — max drop 68%

Model: `Qwen/Qwen3.5-4B` · Effort: `default` · Label: **non-robust**

Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ be a function such that for all positive integers $m$ and $n$, 
\begin{equation*}
    f(m) + f(n) = f(m + n + mn).
\end{equation*}
Across all functions $f$ such that $f(n) \leq 617$ for all $n \leq 617$, how many different values can $f(3520)$ take?

In [7]:
import pandas as pd
train_df = pd.read_parquet(REPO_ROOT / "data_artifacts/train-main-v2/train_allowed.parquet")

In [8]:
df1 = train_df[train_df["model_id"] == "Qwen/Qwen3.5-4B"]

In [9]:
df1

,model_id,reasoning_effort,problem,is_robust,max_drop
0,Qwen/Qwen3.5-4B,default,The Fibonacci numbers are defined as follows: ...,True,-0.01
1,Qwen/Qwen3.5-4B,default,Find the sum of all real numbers $r$ such that...,True,0.01
2,Qwen/Qwen3.5-4B,default,An isosceles trapezoid has an inscribed circle...,True,0.01
3,Qwen/Qwen3.5-4B,default,Find the sum of all integer bases $b>9$ for wh...,True,0.04
4,Qwen/Qwen3.5-4B,default,Alice writes all positive integers from $1$ to...,None,0.11
5,Qwen/Qwen3.5-4B,default,Let $S$ denote the value of the infinite sum\n...,False,0.27
6,Qwen/Qwen3.5-4B,default,Let $N$ be the number of positive integer divi...,False,0.40


In [10]:
problem_statement = df1.iloc[6]["problem"]
problem_statement

'Let $N$ be the number of positive integer divisors of $17017^{17}$ that leave a remainder of $5$ when divided by $12$. Find $N$.'

In [11]:
# Run without streaming; rerun this cell to compare warmed-up timings.
result = solve(problem_statement, max_new_tokens=1024*8, stream=False)
print({key: value for key, value in result.items() if key != "response"})
print(result["response"])


Generating...
{'generated_tokens': 8192, 'hit_token_limit': True, 'elapsed_seconds': 273.16, 'tokens_per_second': 29.99, 'peak_allocated_gib': 3.38}
Let $n = 17017^{17}$. We want to find the number of positive integer divisors of $n$ that leave a remainder of $5$ when divided by $12$.
Let $d$ be a divisor of $n$. We are looking for the number of such $d$ such that $d \equiv 5 \pmod{12}$.
First, let's find the prime factorization of $17017$.
$17017$ is not divisible by 2 (odd).
Sum of digits is $1+7+0+1+7 = 16$, not divisible by 3.
Ends in 7, not divisible by 5.
$17017 = 17 \times 1001$.
$1001 = 7 \times 11 \times 13$.
So $17017 = 17 \times 7 \times 11 \times 13$.
Let's check: $17 \times 7 = 119$. $119 \times 11 = 1309$. $1309 \times 13 = 17017$. Correct.
So $17017 = 7 \times 11 \times 13 \times 17$.
The prime factors are 7, 11, 13, 17.
The number is $n = (7 \times 11 \times 13 \times 17)^{17} = 7^{17} \times 11^{17} \times 13^{17} \times 17^{17}$.
We are looking for divisors $d$ of $n$